# GETCHA: YOLOv5s vs YOLO26s 재실험 (Colab, T4)

데이터 준비 → 학습 → 속도 측정 → 서버 프로파일링 순서로 실행한다.

## 1. 환경 / 데이터

In [ ]:
from google.colab import auth, drive
auth.authenticate_user()
drive.mount('/content/drive')

DATASET_FILE_ID = ''          # Google Drive의 dataset.zip 파일 ID
DATA_DIR = '/content/getcha_data'
RESULT_DIR = '/content/drive/MyDrive/getcha_results'

In [ ]:
import os, zipfile
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload

# 대용량 파일은 마운트 경로 복사가 끊기는 경우가 있어 Drive API로 분할 다운로드
if not os.path.exists(DATA_DIR):
    req = build('drive', 'v3').files().get_media(fileId=DATASET_FILE_ID, supportsAllDrives=True)
    with open('/content/dataset.zip', 'wb') as f:
        dl, done = MediaIoBaseDownload(f, req, chunksize=200 * 1024 * 1024), False
        while not done:
            status, done = dl.next_chunk(num_retries=10)
    with zipfile.ZipFile('/content/dataset.zip') as z:
        z.extractall(DATA_DIR)
    os.remove('/content/dataset.zip')

In [ ]:
!git clone -q https://github.com/<Ltaerim>/getcha-yolo-upgrade.git /content/repo
!git clone -q https://github.com/ultralytics/yolov5 /content/repo/yolov5
%cd /content/repo
!pip install -q -r requirements.txt -r yolov5/requirements.txt

## 2. 전처리 (라벨 재정의 + 416px 사전 리사이즈)

In [ ]:
import glob
DATA_ROOT = os.path.dirname(os.path.dirname(glob.glob(f'{DATA_DIR}/**/images/train', recursive=True)[0]))
!python scripts/prepare_data.py --root {DATA_ROOT} --img 416 --bench-dir bench_original

## 3. 학습 (YOLOv5s / YOLO26s, 동일 조건)

In [ ]:
%env WANDB_MODE=disabled
!python scripts/train.py --data configs/data.yaml --project {RESULT_DIR}/runs --epochs 30 --out results/accuracy.csv

## 4. 추론 속도 / 서버 프로파일링

In [ ]:
!python scripts/benchmark.py --v5 {RESULT_DIR}/runs/v5/weights/best.pt --new {RESULT_DIR}/runs/yolo26s/weights/best.pt \
    --images bench_original --yolov5-dir yolov5 --out results/speed.csv
sample = sorted(glob.glob('bench_original/*'))[0]
!python scripts/profile_server.py --weights {RESULT_DIR}/runs/v5/weights/best.pt --image "{sample}" --yolov5-dir yolov5

In [ ]:
!python scripts/plot_results.py
!cp -r results {RESULT_DIR}/